# Hydrologic indices: TWI, SPI, STI (W-12 → W-14)

Three classic catchment-scale indices computed from `accumulation` × `slope`:

| Index | Formula | Use |
|-------|---------|-----|
| **TWI** (W-12) | `ln(SCA / tan(slope))` | Wetness / saturation likelihood (Beven & Kirkby 1979) |
| **SPI** (W-13) | `SCA × tan(slope)` | Erosion / stream-power proxy |
| **STI** (W-14) | `(SCA/22.13)^0.6 × (sin(slope)/0.0896)^1.3` | Sediment transport (Moore & Burch 1986) |

All three share the `_area_slope_index` kernel inside `DEM` — caller supplies a precomputed slope
raster (in degrees) or lets the method compute it from `DEM.slope()`.

In [ ]:
import numpy as np
from pyramids.dataset import Dataset
from digitalrivers import DEM

# Catchment with a clear main-stem chain and tributaries.
rows, cols = 15, 15
z = np.full((rows, cols), 100.0, dtype=np.float32)
for r in range(rows):
    z[r, 7] = float(40 - 2.5 * r)
for c in range(7):
    z[5, c] = float(30 - c)
for c in range(8, cols):
    z[10, c] = float(20 - (c - 8))

ds = Dataset.create_from_array(
    z, top_left_corner=(0.0, 0.0), cell_size=30.0,
    epsg=32618, no_data_value=-9999.0,
)
dem = DEM(ds.raster)
filled = dem.fill_depressions(method="priority_flood")
fd = filled.flow_direction(method="d8")
acc = fd.accumulate()
print(f"Accumulation max: {int(acc.read_array().max())} cells")

## W-12 — Topographic Wetness Index (TWI)

TWI = ln(SCA / tan(slope)). High TWI marks cells likely to saturate first — large upstream area
AND low slope. The classic stream-bottom cells score highest.

In [ ]:
twi = filled.twi(acc).read_array()
valid_twi = twi[twi != -9999.0]
valid_twi = valid_twi[np.isfinite(valid_twi)]
print(f"TWI range:     [{valid_twi.min():.3f}, {valid_twi.max():.3f}]")
print(f"Mean TWI:      {valid_twi.mean():.3f}")

## W-13 — Stream Power Index (SPI)

SPI = SCA × tan(slope). Proportional to the rate at which overland flow does work — a proxy for
erosion risk. Steep cells with large upstream area score highest.

In [ ]:
spi = filled.spi(acc).read_array()
valid_spi = spi[(spi != -9999.0) & np.isfinite(spi)]
print(f"SPI range:     [{valid_spi.min():.3f}, {valid_spi.max():.3f}]")
print(f"Mean SPI:      {valid_spi.mean():.3f}")

# SPI must be ≥ 0 (SCA and tan(slope) both non-negative on valid cells)
assert (valid_spi >= 0).all()

## W-14 — Sediment Transport Index (STI)

STI = (SCA / 22.13)^0.6 × (sin(slope) / 0.0896)^1.3 — the per-unit-area form of the RUSLE LS
factor. Predicts where flow will transport sediment rather than deposit it.

In [ ]:
sti = filled.sti(acc).read_array()
valid_sti = sti[(sti != -9999.0) & np.isfinite(sti)]
print(f"STI range:     [{valid_sti.min():.3f}, {valid_sti.max():.3f}]")
print(f"Mean STI:      {valid_sti.mean():.3f}")

assert (valid_sti >= 0).all()

## Reusing a precomputed slope raster

All three methods accept `slope_deg=` as a kwarg to skip recomputation when you already have a
slope raster handy.

In [ ]:
slope_ratio = filled.slope().read_array()  # m/m
slope_deg_arr = np.degrees(np.arctan(np.where(
    np.isfinite(slope_ratio), slope_ratio, 0.0,
)))
slope_deg = Dataset.create_from_array(
    slope_deg_arr.astype(np.float32),
    geo=filled.geotransform, epsg=filled.epsg, no_data_value=-9999.0,
)
twi_reused = filled.twi(acc, slope_deg=slope_deg).read_array()
np.testing.assert_allclose(
    twi_reused[twi_reused != -9999.0],
    twi[twi != -9999.0],
    atol=1e-3,
)
print("TWI with reused slope matches default-computed TWI.")

## Summary

Three classic indices in three one-line calls. The shared `_area_slope_index` kernel handles
no-data masking, slope-floor clamping (to avoid `log(0)` on flats), and float32 output wrapping.